# 🧠 3D 뇌종양 분할 (Keras 3D U-Net) — 실습

카드 `ailab-2026-0002` 의 코드 골격을 Colab에서 직접 돌려본다.
데이터: Medical Segmentation Decathlon **Task01_BrainTumour**(재배포 가능 미러) / BraTS.

원본 예제: https://keras.io/examples/vision/brain_tumor_segmentation/

## 1. 준비

In [ ]:
# === MedKOS AI랩 공통 준비 셀 =====================================
# 1) Google Drive 마운트 (데이터·체크포인트 영구 보관)
from google.colab import drive
drive.mount('/content/drive')
import pathlib
BASE = pathlib.Path('/content/drive/MyDrive/MedKOS')
(BASE/'data').mkdir(parents=True, exist_ok=True)
(BASE/'ckpt').mkdir(parents=True, exist_ok=True)
print('Drive ready at', BASE)

In [ ]:
# 2) 최신 repo 당겨오기 (datasets.py 등 결정론 코드 사용)
import os, sys
REPO = 'https://github.com/ehdbddl06001-ui/my-github-test.git'
if not os.path.exists('/content/medkos'):
    !git clone --depth 1 $REPO /content/medkos
else:
    !cd /content/medkos && git pull --ff-only
sys.path.append('/content/medkos')

In [ ]:
# 4) GPU 확인
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip -q install nibabel
import nibabel, numpy as np, keras
print('nibabel', nibabel.__version__, '| keras', keras.__version__)

## 2. 데이터
MSD Task01을 `BASE/'data'/'msd-brain'` 아래에 둔다(한 번만 내려받아 재사용).
`imagesTr/*.nii.gz`(4채널 MRI) + `labelsTr/*.nii.gz`(마스크).

In [ ]:
import pathlib
DATA = BASE/'data'/'msd-brain'
DATA.mkdir(parents=True, exist_ok=True)
# 받는 법: http://medicaldecathlon.com/ 의 Task01_BrainTumour.tar 를
# 한 번 받아 DATA 아래 풀어둔다. (수 GB → Drive에 두면 다음 세션 재사용)
print('데이터 폴더:', DATA); print(sorted(p.name for p in DATA.glob('*'))[:5])

In [ ]:
import nibabel as nib, numpy as np
def load_case(img_path, lbl_path, size=64):
    img = nib.load(str(img_path)).get_fdata()      # (H,W,D,4)
    lbl = nib.load(str(lbl_path)).get_fdata()       # (H,W,D)
    # z-score 정규화(채널별) + 간단 리사이즈는 생략(패치/크롭 권장)
    img = (img - img.mean()) / (img.std() + 1e-6)
    return img.astype('float32'), (lbl > 0).astype('float32')
# TODO: tf.data.Dataset 으로 패치 샘플링 + 배치 구성

## 3. 3D U-Net 정의
카드의 골격과 동일. 메모리 때문에 patch(64³)·filters를 작게 시작.

In [ ]:
import keras; from keras import layers
def conv_block(x, f):
    x = layers.Conv3D(f, 3, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Conv3D(f, 3, padding='same')(x)
    x = layers.BatchNormalization()(x); return layers.Activation('relu')(x)

def build_unet3d(shape=(64,64,64,4), n_classes=1):
    inp = keras.Input(shape)
    c1 = conv_block(inp, 16); p1 = layers.MaxPooling3D()(c1)
    c2 = conv_block(p1, 32);  p2 = layers.MaxPooling3D()(c2)
    b  = conv_block(p2, 64)
    u2 = layers.concatenate([layers.UpSampling3D()(b), c2]); d2 = conv_block(u2, 32)
    u1 = layers.concatenate([layers.UpSampling3D()(d2), c1]); d1 = conv_block(u1, 16)
    out = layers.Conv3D(n_classes, 1, activation='sigmoid')(d1)
    return keras.Model(inp, out)

model = build_unet3d(); model.summary()

## 4. 손실(Dice) · 학습
종양이 작아 픽셀 정확도는 속기 쉽다 → **Dice**로 겹침을 직접 최적화.

In [ ]:
import keras
def dice_loss(y_true, y_pred, eps=1e-6):
    inter = keras.ops.sum(y_true*y_pred, axis=[1,2,3])
    union = keras.ops.sum(y_true+y_pred, axis=[1,2,3])
    return 1 - keras.ops.mean((2*inter+eps)/(union+eps))

model.compile(optimizer='adam', loss=dice_loss, metrics=['accuracy'])
# TODO: model.fit(train_ds, validation_data=val_ds, epochs=5)
# 끝나면: model.save(BASE/'ckpt'/'brain_unet3d.keras')

## 5. 회고 → 카드에 반영
검증 Dice, 예측 마스크 캡처 1장, 막힌 점을 `ailab-2026-0002`의 `## My notes`에 옮긴다.

**변형 실습**: 손실을 BCE+Dice로, patch를 64³→128³로 바꿔 성능·메모리 비교.